# Simple Batch Injection Demo (10 x 1000)

This notebook scales up the single-run pipeline to a batch experiment:

1. Set up the project path and imports
2. Configure one batch run
3. Load **one** Rubin coadd (same image is reused across iterations)
4. Define a detection function (matched-filter + MCI example)
5. Run **10 iterations** of injection, each adding **1000 synthetic clusters**
   - each iteration creates its own injected image
   - MCI detection is run independently on each iteration's injected image
   - per-iteration injection + detection catalogs are checkpointed to disk
6. Combine all 10 truth catalogs and all 10 detection catalogs into one
   pooled retrieval (`pipe.analyze`) — this is what the final completeness
   plots are computed from
7. Make and save plots

**Effective dataset size:** 10000 injected clusters total, each with a real
PSF and Rubin background, but only one base image is loaded into memory.

In [ ]:
import os
import sys
import inspect
from pathlib import Path

# Auto-detect repo root from this notebook's location.
# Falls back to an explicit path if running outside the repo (e.g. uploaded
# to a fresh RSP user dir) — edit FALLBACK_REPO_ROOT if you need that.
FALLBACK_REPO_ROOT = Path('/home/snair63/WORK/INJECT/star-cluster-injection-pipeline')

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / 'src' / 'config.py').is_file():
            return parent
    if (FALLBACK_REPO_ROOT / 'src' / 'config.py').is_file():
        return FALLBACK_REPO_ROOT.resolve()
    raise RuntimeError(
        'Could not locate src/config.py. Set FALLBACK_REPO_ROOT above to the '
        'folder that directly contains src/.'
    )

repo_root = _find_repo_root()
os.chdir(repo_root)

# Make sure our src/ wins over any installed package of the same name
sys.path = [p for p in sys.path if p != str(repo_root)]
sys.path.insert(0, str(repo_root))

# Drop any previously imported src.* modules so edits to src/ take effect
for name in list(sys.modules.keys()):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]

from lsst.daf.butler import Butler
from src.config import InjectionConfig, ClusterConfig
from src.pipeline import InjectionPipeline
from src.detection import run_cluster_detection
from src.plotting import plot_per_iteration_positions, plot_per_iteration_2d
from src.retrieval import ClusterRetrieval

print('cwd now:', os.getcwd())
print('Using repo_root:', repo_root)
print('InjectionConfig loaded from:', inspect.getsourcefile(InjectionConfig))


## 1. Configure The Batch Run

Most users only edit the values in this cell.

- `band`: Rubin filter to inject into
- `cutout_size`: pixel size of the coadd cutout used for **every** iteration
- `n_clusters`: how many clusters to inject **per iteration**
  (the actual count comes from `N_PER_ITER` in section 4 — this default is
  just used by `pipe.generate_catalog()` if you call it directly)
- `seed`: used as the master seed; iteration `k` uses seed `k`
- `psf_fwhm_fallback`: PSF FWHM to use if the coadd PSF is unavailable
- `output_dir`: where catalogs/checkpoints get written

`ClusterConfig` controls the parameter ranges (profile, magnitude, size,
concentration) that random catalogs are drawn from.

In [ ]:
BUTLER_REPO = 'dp02'
BUTLER_COLLECTIONS = '2.2i/runs/DP0.2'
DATA_ID = {'tract': 3828, 'patch': 24}

config = InjectionConfig(
    run_name='simple_batch_demo',
    band='i',
    cutout_size=1200,
    n_clusters=1000,
    seed=42,
    edge_buffer=50,
    use_actual_psf=True,
    psf_fwhm_fallback=3.5,
    output_dir=str(repo_root / 'plots' / 'simple_batch_demo'),
    cluster_config=ClusterConfig(
        profile_type='king',
        mag_min=20.0,
        mag_max=26.0,
        r_half_min=2.0,
        r_half_max=10.0,
        concentration_min=5.0,
        concentration_max=30.0,
    ),
)

config

## 2. Load One Rubin Coadd

The same coadd is reused as the base image for every iteration. Each
iteration injects fresh clusters into a clean copy of this image.

The next cell shows the loaded coadd so you can verify it visually.

In [ ]:
butler = Butler(BUTLER_REPO, collections=BUTLER_COLLECTIONS)
pipe = InjectionPipeline(config)
pipe.load_data(butler=butler, data_id=DATA_ID)

bbox_x_min, bbox_y_min = pipe.bboxes[config.band]
psf_obj = pipe.psf_objs[config.band]

print(f'Loaded image shape: {pipe.image.shape}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

vmin, vmax = np.percentile(pipe.image, [1, 99])
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(pipe.image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
ax.set_title(f'Loaded {config.band}-band coadd ({pipe.image.shape[1]} x {pipe.image.shape[0]} px)')
ax.set_xlabel('X [pix]')
ax.set_ylabel('Y [pix]')
plt.show()

## 3. Detection Function — **PLUG IN YOUR DETECTOR HERE**

This is the only place in the notebook where the cluster-finding method
lives. `run_batch` calls whatever function you pass as `detector_fn` once
per iteration on that iteration's injected image, so your detector runs
independently on each of the 10 images.

**Replace the body of `my_detector` with your own code.** Anything that
takes a 2D image and returns a list of candidate detections will work.

The example below is a stand-in that calls `run_cluster_detection`
(a matched-filter + MCI implementation shipped in `src/detection.py`) so
the demo produces realistic numbers out of the box — but it is **not**
required by the pipeline.

**Required output format:** `list[dict]` where each dict has at least
`x` and `y` (pixel coordinates). Optional fields like `flux`, `snr`,
`r_half`, `magnitude` unlock extra diagnostics later.


In [ ]:
# ======================================================================
# >>> PLUG IN YOUR DETECTOR HERE <<<
#
# `my_detector(image)` must accept a 2D numpy array and return
# `list[dict]` with at least 'x' and 'y' (pixel coordinates).
# Optional keys ('flux', 'snr', 'r_half', 'magnitude', ...) unlock
# extra diagnostics in section 5.
#
# The body below is just a *stand-in* using the matched-filter + MCI
# detector in src/detection.py so this demo produces real numbers.
# Delete it and call your own code instead.
# ======================================================================
def my_detector(image):
    """Stand-in detector — replace the body with your own method."""
    from src.detection import run_cluster_detection

    return run_cluster_detection(
        image=image,
        psf_fwhm=config.psf_fwhm_fallback,
        threshold_sigma=3.0,
        mci_max=0.85,
        snr_min=3.0,
        r_half_min=1.0,
        ellipticity_max=0.6,
        box_size=64,
        pixel_scale=config.pixel_scale,
        use_multiscale=True,
        use_mci=True,
        verbose=False,
    )


## 4. Run Batch Injection (10 Iterations x 1000 Clusters)

What this cell does, in order:

1. For each of `N_ITERATIONS` iterations:
   - generate a fresh random catalog of `N_PER_ITER` clusters
   - inject them into a copy of the loaded coadd using the real Rubin PSF
   - run `mci_detector` on that iteration's injected image
   - write per-iteration injection + detection catalogs to `checkpoint_dir`
2. Pool all 10 truth catalogs into `pipe.injection_info`
3. Pool all 10 detection catalogs into `pipe.detection_catalog`

So `pipe.injection_info` ends up with ~10000 entries and
`pipe.detection_catalog` holds every detection MCI made across all 10 images.
Combined retrieval and completeness in section 5 use these pooled lists.

**Knobs you might tune:**
- `N_ITERATIONS`, `N_PER_ITER`
- `n_workers`: number of parallel iterations (1 = sequential / safest)
- `store_images`: keep all 10 injected images in memory (False saves memory)
- `use_psf_cache`: reuse PSF stamps across iterations (big speedup)

In [ ]:
N_ITERATIONS = 10
N_PER_ITER   = 1000

iterations = pipe.run_batch(
    n_iterations=N_ITERATIONS,
    n_per_iter=N_PER_ITER,
    psf_obj=psf_obj,
    bbox_x_min=bbox_x_min,
    bbox_y_min=bbox_y_min,
    psf_fwhm_fallback=config.psf_fwhm_fallback,
    detector_fn=mci_detector,
    store_images=False,
    checkpoint_dir=str(repo_root / 'plots' / 'simple_batch_demo' / 'checkpoints'),
    n_workers=1,
    use_psf_cache=True,
    psf_cache_grid=8,
    psf_cache_size=2000,
    band=config.band,
    verbose=True,
)

print(f'Iterations completed: {len(iterations)}')
print(f'Total injected (pooled): {len(pipe.injection_info)}')
print(f'Total detections (pooled): {len(pipe.detection_catalog)}')

## 5. Combined Retrieval And Completeness

`pipe.analyze(match_radius=5.0)` matches the **pooled** detections to the
**pooled** truth catalog (10000 injected, all 10 iterations). Everything
downstream (completeness curves, 50% limits, 2D map) uses this pooled
result, which is the whole point of the 10x1000 design — it gives you
much smoother completeness statistics than a single 10000-cluster run on
one image would.

In [ ]:
stats = pipe.analyze(match_radius=5.0)
stats

## 6. Display And Save Settings

Set these once and the plotting cell below will use them.

- `SHOW_PLOTS_INTERACTIVE`:
  - `True`  → render plots inline in the notebook **and** save to disk
  - `False` → only save plots to disk (recommended for big batches)
- `PLOT_OUTPUT_DIR`: directory for all saved figures
- `PLOTS_TO_MAKE`: which plots to produce
- `N_STAMPS_TO_SAVE`: how many postage stamp triptychs to save
  (set to `len(pipe.injection_info)` to save **all 10000**)

Available plot keys:

`'injection_summary'`, `'before_after'`, `'position_map'`,
`'completeness_1d'`, `'completeness_2d'`, `'psf_fwhm_hist'`, `'psf_fwhm_map'`

**Note:** all selected plots and all postage stamps are saved to disk
regardless of `SHOW_PLOTS_INTERACTIVE`.

In [ ]:
# ---- USER CONTROLS ---------------------------------------------------
SHOW_PLOTS_INTERACTIVE = False  # batches are large; default to save-only
PLOT_OUTPUT_DIR        = str(Path(config.output_dir) / 'plots')

PLOTS_TO_MAKE = [
    'injection_summary',   # original + injected + diff + mag/size/flux distributions
    'before_after',        # 3-panel original / injected / difference (last iter image)
    'position_map',        # detected vs missed cluster positions (pooled)
    'completeness_1d',     # completeness vs magnitude AND vs r_half (pooled)
    'completeness_2d',     # 2D completeness heatmap (mag x r_half)
    'psf_fwhm_hist',       # PSF FWHM at each injection position (pooled)
]

# Save EVERY postage stamp from the pooled injection catalog (all 10000).
# Lower this if you only want a sample.
N_STAMPS_TO_SAVE = len(pipe.injection_info)
# ----------------------------------------------------------------------

Path(PLOT_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('Plots will be saved to:', PLOT_OUTPUT_DIR)
print('Interactive display:', SHOW_PLOTS_INTERACTIVE)
print('Postage stamps to save:', N_STAMPS_TO_SAVE)

results = pipe.make_plots(
    output_dir=PLOT_OUTPUT_DIR,
    plots=PLOTS_TO_MAKE,
    show=SHOW_PLOTS_INTERACTIVE,
    save=True,
    n_stamps=N_STAMPS_TO_SAVE,
)

print('Saved files:')
for k, v in results['saved'].items():
    print(f' - {k}: {v}')

## 7. Optional: Per-Iteration Diagnostics

These plots break the pooled result back out by iteration so you can see
how individual injection runs behaved (e.g. iteration-to-iteration scatter
in completeness or position recovery). Useful for debugging — skip if you
only care about the pooled completeness from section 6.

In [ ]:
_ = plot_per_iteration_positions(
    iterations=iterations,
    base_image=pipe.image,
    ClusterRetrieval=ClusterRetrieval,
    n_show=4,
    match_radius=5.0,
)

_ = plot_per_iteration_2d(
    iterations=iterations,
    config=config,
    ClusterRetrieval=ClusterRetrieval,
    n_show=4,
    match_radius=5.0,
)